# MetaEngine GPU Worker (Colab T4)

This notebook connects a free Google Colab T4 GPU (16 GB) to the MetaEngine
Ray cluster running on the sandbox machine. Once connected, the GPU
accelerates BoTorch GP fitting by 10-50× compared to CPU.

**Prerequisites:**
1. The sandbox machine must be running `ray start --head --port=6379`
2. An ngrok tunnel must expose port 6379 (see scripts/launch_ngrok_tunnel.sh)
3. Copy the ngrok URL (e.g. `tcp://0.tcp.ngrok.io:12345`) below

**Free tier limits:**
- 12 hour session (auto-disconnects)
- T4 GPU: 16 GB VRAM (plenty for BoTorch GP)
- Re-runs automatically every 12 hours via the loop at the bottom

**To start:**
1. Open this notebook in Colab (Runtime → Change runtime type → T4 GPU)
2. Set `RAY_ADDRESS` below to your ngrok URL
3. Run all cells (Runtime → Run all)
4. The worker runs in the background for 12 hours

In [ ]:
# === CONFIGURATION ===
# Replace this with the ngrok URL from your sandbox machine
# Format: tcp://X.tcp.ngrok.io:PORT
RAY_ADDRESS = "tcp://0.tcp.ngrok.io:12345"  # ← REPLACE THIS

# Number of CPUs to expose (Colab gives 2-4)
NUM_CPUS = 2

# Auto-reconnect every 11 hours (before 12h disconnect)
AUTO_RECONNECT = True
RECONNECT_INTERVAL_HOURS = 11

print(f"Ray address: {RAY_ADDRESS}")
print(f"CPUs: {NUM_CPUS}, GPU: T4 16GB")
print(f"Auto-reconnect: {AUTO_RECONNECT} (every {RECONNECT_INTERVAL_HOURS}h)")

In [ ]:
# === VERIFY GPU IS AVAILABLE ===
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU detected:")
    print(result.stdout[:500])
else:
    print("✗ No GPU! Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU not available")

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# === INSTALL DEPENDENCIES ===
# (Takes ~2 minutes on first run; cached after that)
import sys
print("Installing Ray + BoTorch...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ray',
    'botorch',
    'gpytorch',
    'pyngrok',
])
print("✓ All dependencies installed")

# Verify
import ray
import botorch
print(f"Ray: {ray.__version__}")
print(f"BoTorch: {botorch.__version__}")

In [ ]:
# === CLONE METAENGINE REPO (for the worker code) ===
import os
import subprocess

REPO_URL = "https://github.com/PatrickFrome/EngineTest.git"
REPO_DIR = "/content/EngineTest"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
    print("✓ Repo cloned")
else:
    print("Repo already exists, pulling latest...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Install metaengine package
os.chdir(f"{REPO_DIR}/METAENGINE_SLICE3_RESTORED")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
print("✓ MetaEngine installed")

# Set env var so the worker finds ROOT
os.environ['ME_BENCHMARK_ROOT'] = f"{REPO_DIR}/METAENGINE_SLICE3_RESTORED"

In [ ]:
# === CONNECT TO RAY CLUSTER ===
import ray

print(f"Connecting to Ray cluster at {RAY_ADDRESS}...")
ray.init(
    address=RAY_ADDRESS,
    num_cpus=NUM_CPUS,
    num_gpus=1,  # Expose the T4 GPU to the cluster
    ignore_reinit_error=True,
    log_to_driver=False,
)

# Verify connection
resources = ray.cluster_resources()
print("\n✓ Connected to Ray cluster!")
print(f"Cluster resources: {resources}")
if 'GPU' in resources:
    print(f"GPUs available: {resources['GPU']}")
else:
    print("⚠ No GPU visible to cluster (check driver node)")

In [ ]:
# === GPU TASK DEFINITION ===
# This remote function runs on the Colab GPU worker.
# It fits a BoTorch GP using CUDA — 10-50x faster than CPU.

@ray.remote(num_gpus=1)
def fit_gp_on_gpu(X_list, Y_list):
    """Fit a SingleTaskGP on GPU. Returns model state dict.

    X_list: list of feature vectors (each is list[float])
    Y_list: list of fitness values (each is float)
    """
    import torch
    import botorch
    from botorch.models import SingleTaskGP
    from gpytorch.mlls import ExactMarginalLogLikelihood
    from botorch.fit import fit_gpytorch_mll

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[gpu-worker] fitting GP on {device}, n={len(X_list)} observations")

    X = torch.tensor(X_list, dtype=torch.double, device=device)
    Y = torch.tensor([[y] for y in Y_list], dtype=torch.double, device=device)

    t0 = time.time()
    gp = SingleTaskGP(X, Y)
    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)
    elapsed = time.time() - t0

    # Return model state (CPU tensors for serialization)
    state = {k: v.cpu() for k, v in gp.state_dict().items()}
    print(f"[gpu-worker] GP fit in {elapsed:.2f}s on {device}")
    return state

# Test that the GPU worker is reachable
fut = fit_gp_on_gpu.remote([[1.0, 2.0], [3.0, 4.0]], [0.5, 0.6])
test_state = ray.get(fut, timeout=60)
print(f"✓ GPU worker test passed — GP state dict has {len(test_state)} keys")

In [ ]:
# === KEEP ALIVE LOOP ===
# This cell runs forever (until Colab disconnects at 12h).
# It periodically prints a heartbeat so Colab doesn't kill us.

import time
import sys

print(f"=== GPU WORKER ACTIVE ===")
print(f"Connected to: {RAY_ADDRESS}")
print(f"GPU: T4 16GB")
print(f"Heartbeat every 60s")
print(f"Colab will disconnect in ~12 hours — re-run notebook to reconnect")
print(f"========================================")
print()

heartbeat_count = 0
start_time = time.time()

try:
    while True:
        heartbeat_count += 1
        elapsed = time.time() - start_time
        resources = ray.cluster_resources()
        gpu_count = resources.get('GPU', 0)
        print(f"[heartbeat {heartbeat_count}] alive {elapsed:.0f}s, "
              f"cluster GPUs={gpu_count}, cluster CPUs={resources.get('CPU', 0)}",
              flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopped by user")
except Exception as e:
    print(f"Error: {e}")
    print("Ray cluster may have disconnected. Re-run this notebook.")
finally:
    ray.shutdown()

## 📋 Setup Instructions

### On the sandbox machine (one-time setup):

```bash
# 1. Install ngrok (if not installed)
snap install ngrok

# 2. Get your ngrok auth token (free at https://dashboard.ngrok.com)
ngrok config add-authtoken YOUR_TOKEN

# 3. Start Ray head node on the sandbox
pip install ray
ray start --head --port=6379 --num-cpus=4

# 4. Expose Ray port via ngrok
ngrok tcp 6379
# → Copy the forwarding URL (e.g. tcp://0.tcp.ngrok.io:12345)
```

### On Colab:
1. Upload this notebook to Google Drive
2. Open in Colab → Runtime → Change runtime type → T4 GPU
3. Set `RAY_ADDRESS` in the config cell to your ngrok URL
4. Runtime → Run all

The GPU worker will join the Ray cluster and accelerate BoTorch GP fitting
for the next 12 hours. Re-run the notebook to reconnect.